In [ ]:
# Cell 1 - Structural profile, missingness, and key uniqueness

from pathlib import Path
import pandas as pd

train_path = Path("train.csv")
test_path = Path("test.csv")
sub_path = Path("sample_submission.csv")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
sub_df = pd.read_csv(sub_path)

print(f"train shape: {train_df.shape}")
print(f"test shape:  {test_df.shape}")
print(f"submission shape: {sub_df.shape}")

print(f"train id unique: {train_df['id'].nunique() == len(train_df)} ({train_df['id'].nunique()}/{len(train_df)})")
print(f"test id unique:  {test_df['id'].nunique() == len(test_df)} ({test_df['id'].nunique()}/{len(test_df)})")

print("\ntrain missing:")
print(train_df.isna().sum())
print("\ntest missing:")
print(test_df.isna().sum())

print("\ntrain types:")
print(train_df.dtypes)

In [ ]:
# Cell 2 - Target listPrice distribution, skewness, and naive baseline

import numpy as np

prices = train_df["listPrice"]

min_p = prices.min()
max_p = prices.max()
mean_p = prices.mean()
median_p = prices.median()
std_p = prices.std()
skew_p = prices.skew()

q01, q05, q25, q50, q75, q95, q99 = prices.quantile([0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])
iqr_p = q75 - q25

# Naive baseline: predicting median price minimizes MAE
naive_mae = np.mean(np.abs(prices - median_p))

print(f"count: {len(prices)}")
print(f"min: {min_p:,.2f} | max: {max_p:,.2f}")
print(f"mean: {mean_p:,.2f} | median: {median_p:,.2f} | std: {std_p:,.2f}")
print(f"skewness: {skew_p:.4f}")
print(f"non-positive prices: {(prices <= 0).sum()}")
print("\npercentiles:")
print(f" 1%:  {q01:,.2f}")
print(f" 5%:  {q05:,.2f}")
print(f"25%:  {q25:,.2f}")
print(f"50%:  {q50:,.2f}")
print(f"75%:  {q75:,.2f}")
print(f"95%:  {q95:,.2f}")
print(f"99%:  {q99:,.2f}")
print(f"IQR:  {iqr_p:,.2f}")
print(f"\nnaive median-predictor train MAE: {naive_mae:,.2f}")

In [ ]:
# Cell 3 - Text length profiles and representative sample inspection

train_char_len = train_df["text"].str.len()
test_char_len = test_df["text"].str.len()
train_word_len = train_df["text"].str.split().str.len()
test_word_len = test_df["text"].str.split().str.len()

print("Character lengths:")
print(f"train - min: {train_char_len.min()}, median: {train_char_len.median():.0f}, mean: {train_char_len.mean():.1f}, max: {train_char_len.max()}")
print(f"test  - min: {test_char_len.min()}, median: {test_char_len.median():.0f}, mean: {test_char_len.mean():.1f}, max: {test_char_len.max()}")

print("\nWord counts:")
print(f"train - min: {train_word_len.min()}, median: {train_word_len.median():.0f}, mean: {train_word_len.mean():.1f}, max: {train_word_len.max()}")
print(f"test  - min: {test_word_len.min()}, median: {test_word_len.median():.0f}, mean: {test_word_len.mean():.1f}, max: {test_word_len.max()}")

# Sample across low, median, and high price tiers
p_low = train_df[train_df["listPrice"] <= 100000].iloc[0]
p_mid = train_df[(train_df["listPrice"] >= 490000) & (train_df["listPrice"] <= 510000)].iloc[0]
p_high = train_df[train_df["listPrice"] >= 5000000].iloc[0]

print("\nSample low price (<= 100k):")
print(f"id: {p_low['id']} | price: {p_low['listPrice']:,.2f}")
print(f"text preview: {p_low['text'][:250]}...\n")

print("Sample median price (~500k):")
print(f"id: {p_mid['id']} | price: {p_mid['listPrice']:,.2f}")
print(f"text preview: {p_mid['text'][:250]}...\n")

print("Sample high price (>= 5M):")
print(f"id: {p_high['id']} | price: {p_high['listPrice']:,.2f}")
print(f"text preview: {p_high['text'][:250]}...")

In [ ]:
# Cell 4 - Structured attribute extraction yields and correlation with target

import re
import pandas as pd
import numpy as np

def extract_specs(text_series):
    re_beds = re.compile(r'\b(\d+(?:\.\d+)?)\s*(?:beds?|bedrooms?|bds?|br)\b', re.I)
    re_baths = re.compile(r'\b(\d+(?:\.\d+)?)\s*(?:baths?|bathrooms?|ba)\b', re.I)
    re_sqft = re.compile(r'\b([\d,]+)\s*(?:sq(?:uare)?\.?\s*(?:feet|ft)|sqft)\b', re.I)
    re_acres = re.compile(r'\b([\d,]+(?:\.\d+)?)\s*(?:acres?|ac)\b', re.I)
    re_garage = re.compile(r'\b(\d+)\s*(?:car|garage)\b', re.I)
    
    beds, baths, sqft, acres, garage = [], [], [], [], []
    
    for t in text_series:
        m_bed = re_beds.search(t)
        beds.append(float(m_bed.group(1)) if m_bed else np.nan)
        
        m_bath = re_baths.search(t)
        baths.append(float(m_bath.group(1)) if m_bath else np.nan)
        
        m_sqft = re_sqft.search(t)
        if m_sqft:
            try:
                sqft.append(float(m_sqft.group(1).replace(',', '')))
            except ValueError:
                sqft.append(np.nan)
        else:
            sqft.append(np.nan)
            
        m_ac = re_acres.search(t)
        if m_ac:
            try:
                acres.append(float(m_ac.group(1).replace(',', '')))
            except ValueError:
                acres.append(np.nan)
        else:
            acres.append(np.nan)
            
        m_gar = re_garage.search(t)
        garage.append(float(m_gar.group(1)) if m_gar else np.nan)
        
    return pd.DataFrame({
        "beds": beds,
        "baths": baths,
        "sqft": sqft,
        "acres": acres,
        "garage": garage
    }, index=text_series.index)

extracted_train = extract_specs(train_df["text"])
extracted_test = extract_specs(test_df["text"])

log_price = np.log1p(train_df["listPrice"])

print("Extraction yield / coverage on train:")
for col in extracted_train.columns:
    count = extracted_train[col].notna().sum()
    pct = count / len(extracted_train) * 100
    valid_mask = extracted_train[col].notna()
    corr = extracted_train.loc[valid_mask, col].corr(log_price.loc[valid_mask], method="spearman")
    print(f"{col:8s}: {count:5d}/{len(extracted_train)} ({pct:5.1f}%) | Spearman corr with log(price): {corr:+.3f}")

print("\nExtraction yield / coverage on test:")
for col in extracted_test.columns:
    count = extracted_test[col].notna().sum()
    pct = count / len(extracted_test) * 100
    print(f"{col:8s}: {count:5d}/{len(extracted_test)} ({pct:5.1f}%)")

In [ ]:
# Cell 5 - N-gram associations and lexical drivers of listing price

from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import numpy as np

tfidf = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=50,
    max_features=5000,
    sublinear_tf=True
)

X_tfidf = tfidf.fit_transform(train_df["text"])
vocab = np.array(tfidf.get_feature_names_out())

log_p = np.log1p(train_df["listPrice"]).values
y_diff = log_p - log_p.mean()
y_std = log_p.std()
n = len(log_p)

mean_x = np.array(X_tfidf.mean(axis=0)).ravel()
var_x = np.array(X_tfidf.power(2).mean(axis=0)).ravel() - (mean_x ** 2)
std_x = np.sqrt(np.maximum(var_x, 1e-12))

cov = np.array(X_tfidf.T.dot(y_diff)).ravel() / n
corrs = cov / (std_x * y_std)

corr_series = pd.Series(corrs, index=vocab)

print(f"Vocabulary size (min_df=50, max_features=5000): {len(vocab)}")
print("\nTop 15 tokens associated with HIGHER price:")
for token, val in corr_series.nlargest(15).items():
    print(f"  {token:25s}: {val:+.4f}")

print("\nTop 15 tokens associated with LOWER price:")
for token, val in corr_series.nsmallest(15).items():
    print(f"  {token:25s}: {val:+.4f}")

In [ ]:
# Cell 6 - Property archetype segmentation and price regime analysis

import re
import pandas as pd
import numpy as np

# Regex for key property archetypes
re_land = re.compile(r'\b(?:lot|parcel|vacant\s+land|build\s+your\s+dream|raw\s+land)\b', re.I)
re_condo = re.compile(r'\b(?:condo|condominium|high-floor|co-op)\b', re.I)
re_luxury = re.compile(r'\b(?:penthouse|estate|waterfront|chef\s+kitchen|marble)\b', re.I)

is_land = train_df["text"].str.contains(re_land) & (extracted_train["beds"].isna() | (extracted_train["beds"] == 0))
is_condo = train_df["text"].str.contains(re_condo) & ~is_land
is_luxury = train_df["text"].str.contains(re_luxury) & ~is_land

archetype = pd.Series("Single-Family / General", index=train_df.index)
archetype[is_land] = "Vacant Land / Lot"
archetype[is_condo] = "Condominium"
archetype[is_luxury & ~is_condo] = "Luxury Estate"

train_df_archetype = train_df.assign(archetype=archetype)

stats = train_df_archetype.groupby("archetype")["listPrice"].agg(
    count="count",
    median="median",
    mean="mean",
    p25=lambda x: x.quantile(0.25),
    p75=lambda x: x.quantile(0.75)
).reset_index()

stats["pct"] = stats["count"] / len(train_df) * 100
stats = stats.sort_values(by="median")

print("Property Archetype Price Breakdown:")
for _, row in stats.iterrows():
    print(f"  {row['archetype']:25s} | count: {int(row['count']):5d} ({row['pct']:4.1f}%) | "
          f"p25: {row['p25']:>10,.0f} | median: {row['median']:>10,.0f} | p75: {row['p75']:>10,.0f} | mean: {row['mean']:>11,.0f}")


In [ ]:
# Cell 7 - Geographic entity extraction and regional price variation

import re
import pandas as pd
import numpy as np

# Map of major US states frequently appearing in listings
states_map = {
    'OR': r'\b(?:OR|Oregon|Portland|Bend|Gresham|Salem|Eugene|Medford|Klamath)\b',
    'NY': r'\b(?:NY|New York|Manhattan|Brooklyn|Queens|Bronx|Long Island)\b',
    'CA': r'\b(?:CA|California|Los Angeles|San Francisco|San Diego|Sacramento)\b',
    'FL': r'\b(?:FL|Florida|Miami|Orlando|Tampa|Jacksonville)\b',
    'WA': r'\b(?:WA|Washington|Seattle|Spokane|Tacoma|Vancouver)\b',
    'TX': r'\b(?:TX|Texas|Austin|Dallas|Houston|San Antonio)\b',
    'AZ': r'\b(?:AZ|Arizona|Phoenix|Tucson|Scottsdale)\b',
    'NC': r'\b(?:NC|North Carolina|Charlotte|Raleigh)\b',
    'CO': r'\b(?:CO|Colorado|Denver|Boulder|Colorado Springs)\b'
}

def detect_region(text_series):
    region = pd.Series('Other / Unidentified', index=text_series.index)
    for state_code, pattern in states_map.items():
        mask = text_series.str.contains(re.compile(pattern, re.I)) & (region == 'Other / Unidentified')
        region[mask] = state_code
    return region

train_region = detect_region(train_df['text'])
test_region = detect_region(test_df['text'])

train_df_geo = train_df.assign(region=train_region)

geo_stats = train_df_geo.groupby('region')['listPrice'].agg(
    count='count',
    median='median',
    mean='mean',
    p25=lambda x: x.quantile(0.25),
    p75=lambda x: x.quantile(0.75)
).reset_index()

geo_stats['pct_train'] = geo_stats['count'] / len(train_df) * 100
test_counts = test_region.value_counts(normalize=True) * 100
geo_stats['pct_test'] = geo_stats['region'].map(test_counts)
geo_stats = geo_stats.sort_values(by='count', ascending=False)

print('Geographic Distribution and Price Breakdown:')
for _, row in geo_stats.iterrows():
    print(f"  {row['region']:20s} | train: {int(row['count']):5d} ({row['pct_train']:4.1f}%) | test: ({row['pct_test']:4.1f}%) | "
          f"median: {row['median']:>10,.0f} | mean: {row['mean']:>11,.0f}")
